# SARF AraBERT — E0

**Owner:** A  
**Condition:** E0  
**CPT assignment:** None.

No CPT. Fine-tune a fresh AraBERT base checkpoint directly on labelled MSA train, select the checkpoint by Macro-F1 on MSA validation.

This notebook contains no model-training implementation. It invokes the one shared `arabert_pipeline.py`, which reads the one shared AraBERT config and the one global 77-label mapping from the project Drive. The held-out Saudi test is not loaded in this notebook.


## 1. Install the fixed environment

Choose a GPU runtime before running this cell. If Colab asks to restart after installation, restart the runtime, then rerun this cell and the following cells in order.


In [9]:
%pip install --upgrade --prefer-binary "tokenizers==0.21.4" "transformers==4.48.3" "accelerate==1.2.1" "datasets==3.2.0" "scikit-learn==1.6.1" "sentencepiece>=0.2.1"
%pip install --upgrade "PyArabic==0.6.15" "farasapy==0.1.1" "emoji==1.4.2"
%pip install --upgrade --no-deps "arabert==1.0.1"


## 2. Mount Drive and verify the shared AraBERT source

The project Drive must contain exactly one shared pipeline, config, and label mapping. This cell does not load training data, CPT data, or the held-out test.


In [10]:
from google.colab import drive
from pathlib import Path
import json
import torch

drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT")
PIPELINE_PATH = PROJECT_ROOT / "src_04" / "arabert_pipeline.py"
CONFIG_PATH = PROJECT_ROOT / "02_configs" / "arabert_config.json"
MAPPING_PATH = PROJECT_ROOT / "02_configs" / "arabert_label_mapping.json"

assert torch.cuda.is_available(), "GPU required; do not run AraBERT training on CPU."
assert PROJECT_ROOT.is_dir(), f"Project folder not found: {PROJECT_ROOT}"
assert PIPELINE_PATH.is_file(), f"Pipeline not found: {PIPELINE_PATH}"
assert CONFIG_PATH.is_file(), f"Config not found: {CONFIG_PATH}"
assert MAPPING_PATH.is_file(), f"Mapping not found: {MAPPING_PATH}"

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
mapping = json.loads(MAPPING_PATH.read_text(encoding="utf-8"))

assert config["owner"] == "A"
assert config["official_conditions"] == ["E0", "E1", "E2", "E3", "EB"]
assert config["official_seeds"] == [42, 123, 2026]
assert mapping["owner"] == "A"
assert mapping["num_labels"] == 77
assert len(mapping["label_to_id"]) == 77

print({
    "status": "shared AraBERT source verified",
    "owner": config["owner"],
    "condition": "E0",
    "gpu": torch.cuda.get_device_name(0),
    "pipeline": str(PIPELINE_PATH),
    "config": str(CONFIG_PATH),
    "mapping": str(MAPPING_PATH),
    "official_seeds": config["official_seeds"],
    "num_labels": mapping["num_labels"],
    "cpt_assignment": config["conditions"]["E0"],
})


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'status': 'shared AraBERT source verified', 'owner': 'A', 'condition': 'E0', 'gpu': 'Tesla T4', 'pipeline': '/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/src_04/arabert_pipeline.py', 'config': '/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/02_configs/arabert_config.json', 'mapping': '/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/02_configs/arabert_label_mapping.json', 'official_seeds': [42, 123, 2026], 'num_labels': 77, 'cpt_assignment': {'description': 'No Saudi-style CPT; fine-tune the base AraBERT checkpoint directly on MSA.', 'cpt_enabled': False, 'cpt': None}}


## 3. Run E0 one seed at a time

Run only one seed per execution. The approved sequence within this condition is `42`, then `123`, then `2026`. The shared pipeline creates outputs directly under `05_runs/arabert/E0/seed_<seed>` and `06_models_and_checkpoints/arabert/E0/seed_<seed>`, refusing to overwrite a completed run.


In [11]:
SEED = 42
assert SEED in [42, 123, 2026]

!python "{PIPELINE_PATH}" --project-root "{PROJECT_ROOT}" --condition "E0" --seed {SEED}


2026-09-14 00:59:36.518471: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[2026-09-14 00:59:44,871 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.
Map: 100% 10732/10732 [00:01<00:00, 8276.33 examples/s]
Map: 100% 1229/1229 [00:00<00:00, 6588.18 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/src_04/arabert_pipeline.py:328: FutureWarnin

In [12]:
SEED = 123
assert SEED in [42, 123, 2026]

!python "{PIPELINE_PATH}" --project-root "{PROJECT_ROOT}" --condition "E0" --seed {SEED}


2026-09-14 01:21:43.417071: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[2026-09-14 01:21:52,067 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.
Map: 100% 10732/10732 [00:00<00:00, 11004.34 examples/s]
Map: 100% 1229/1229 [00:00<00:00, 10825.54 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/src_04/arabert_pipeline.py:328: FutureWarn

In [13]:
SEED = 2026
assert SEED in [42, 123, 2026]

!python "{PIPELINE_PATH}" --project-root "{PROJECT_ROOT}" --condition "E0" --seed {SEED}


2026-09-14 01:53:28.360850: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[2026-09-14 01:53:35,833 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.
Map: 100% 10732/10732 [00:00<00:00, 11213.79 examples/s]
Map: 100% 1229/1229 [00:00<00:00, 11163.47 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/src_04/arabert_pipeline.py:328: FutureWarn